# 1. Clean Data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
%matplotlib inline 
import matplotlib.pyplot as plt
import plotly.express as px
import sklearn

In [2]:
cd C:\Users\Lenovo\OneDrive\Desktop

C:\Users\Lenovo\OneDrive\Desktop


In [6]:
data = pd.read_csv("311data.csv")
data.shape

(800000, 29)

In [7]:
data.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'create_date_utc', 'last_action_et',
       'last_action_utc', 'closed_date_et', 'closed_date_utc', 'origin',
       'street', 'cross_street', 'street_id', 'cross_street_id', 'city',
       'neighborhood', 'census_tract', 'council_district', 'ward',
       'police_zone', 'latitude', 'longitude', 'geo_accuracy'],
      dtype='object')

In [72]:
data.isnull().sum()

_id                       0
group_id                  0
num_requests              0
parent_closed             0
status_name               0
status_code               0
dept                   4103
request_type_name         0
request_type_id           0
create_date_et            0
create_date_utc           0
last_action_et            0
last_action_utc           0
closed_date_et        89049
closed_date_utc       89049
origin                    0
street               308423
cross_street         698878
street_id            301948
cross_street_id      301948
city                      0
neighborhood          36109
census_tract         218417
council_district      34677
ward                  35943
police_zone           36202
latitude              30175
longitude             30175
geo_accuracy              0
dtype: int64

## (1) Drop columns

In [8]:
data = data.drop(columns=['create_date_utc', 
                          'last_action_utc', 
                          'closed_date_utc',
                          'cross_street', 
                          'street', 
                          'street_id', 
                          'cross_street_id',
                         'census_tract'])

In [9]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         36109
council_district     34677
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (2) convert time

In [10]:
data['create_date_et'] = pd.to_datetime(data['create_date_et'])

In [11]:
data['closed_date_et'] = pd.to_datetime(data['closed_date_et'])

In [12]:
data = data.sort_values(by='create_date_et', ascending=True)

In [13]:
data['create_date_et']

151382   2015-04-20 07:37:00
173037   2015-04-20 07:39:00
196183   2015-04-20 07:40:00
111096   2015-04-20 07:41:00
413448   2015-04-20 07:46:00
                 ...        
799997   2024-12-18 09:28:00
799994   2024-12-18 09:32:00
799993   2024-12-18 09:39:00
799990   2024-12-18 09:40:00
799999   2024-12-18 09:41:00
Name: create_date_et, Length: 800000, dtype: datetime64[ns]

## (3) Fill in neighbourhood

In [14]:
# Reverse geocoding function
def get_neighborhood(lat, lon):
    try:
        location = geolocator.reverse(f"{lat}, {lon}", exactly_one=True)
        return location.raw['address'].get('neighbourhood', 'Unknown')
    except:
        return 'Unknown'

#Execute only on data that lacks neighborhood and has longitude and latitude
mask = data['neighborhood'].isna() & data['latitude'].notna() & data['longitude'].notna()
data.loc[mask, 'neighborhood'] = data[mask].apply(
    lambda row: get_neighborhood(row['latitude'], row['longitude']), axis=1
)

In [15]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         30155
council_district     34677
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (4) fill in concil_district

In [16]:
data['council_district'].unique()

array([ 7., nan,  3.,  8.,  9.,  2.,  1.,  5.,  6.,  4.])

In [17]:
mapping = {
    "Knoxville": 3,
    "Regent Square": 5,
    "Carrick": 4,
    "Fairywood": 2,
    "Oakwood": 2,
    "Lincoln Place": 5,
    "Brookline": 4,
    "East Hills": 9,
    "Westwood": 2,
    "Point Breeze": 8,
    "Homewood South": 9,
    "Overbrook": 4,
    "Spring Garden": 1,
    "Beechview": 4,
    "Perry North": 1,
    "Point Breeze North": 8,
    "Squirrel Hill North": 8,
    "Arlington": 3,
    "East Carnegie": 2,
    "East Liberty": 9,
    "North Oakland": 8,
    "Strip District": 7,
    "Summer Hill": 1}

In [18]:
missing_mask = data['council_district'].isna()

data.loc[missing_mask, 'council_district'] = data.loc[missing_mask, 'neighborhood'].map(mapping)

In [19]:
data.isnull().sum()

_id                      0
group_id                 0
num_requests             0
parent_closed            0
status_name              0
status_code              0
dept                  4103
request_type_name        0
request_type_id          0
create_date_et           0
last_action_et           0
closed_date_et       89049
origin                   0
city                     0
neighborhood         30155
council_district     34532
ward                 35943
police_zone          36202
latitude             30175
longitude            30175
geo_accuracy             0
dtype: int64

## (5) parent_closed

In [22]:
data['parent_closed'] = data['parent_closed'].replace({'t': 1, 'f': 0})

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6640\3555678057.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['parent_closed'] = data['parent_closed'].replace({'t': 1, 'f': 0})


In [23]:
data['parent_closed']

151382    1
173037    1
196183    1
111096    1
413448    1
         ..
799997    1
799994    1
799993    1
799990    0
799999    1
Name: parent_closed, Length: 800000, dtype: int64

## (6) Origin

In [144]:
# One-Hot编码origin列
origin_dummies = pd.get_dummies(data['origin'], prefix='origin', dtype=int)

# 合并回原DataFrame
data = pd.concat([data, origin_dummies], axis=1)

# 验证结果
data[['origin', 'origin_Call Center', 'origin_Website']].head()

,origin,origin_Call Center,origin_Website
151382,Call Center,1,0
173037,Call Center,1,0
196183,Call Center,1,0
111096,Call Center,1,0
413448,Call Center,1,0


In [145]:
data.columns

Index(['_id', 'group_id', 'num_requests', 'parent_closed', 'status_name',
       'status_code', 'dept', 'request_type_name', 'request_type_id',
       'create_date_et', 'last_action_et', 'closed_date_et', 'origin', 'city',
       'neighborhood', 'council_district', 'ward', 'police_zone', 'latitude',
       'longitude', 'geo_accuracy', 'origin_Call Center',
       'origin_Control Panel', 'origin_Email', 'origin_QAlert Mobile iOS',
       'origin_Report2Gov Android', 'origin_Report2Gov Website',
       'origin_Report2Gov iOS', 'origin_Text Message', 'origin_Twitter',
       'origin_Website'],
      dtype='object')

## (7) Request_id

In [146]:
name_id_mapping = data.groupby('request_type_name')['request_type_id'].nunique()
name_id_mapping[name_id_mapping > 1]

request_type_name
Crosswalk, New                         2
Dumpster                               2
Earned Income Tax                      2
Electronic/Hazardous Waste Disposal    2
Illegal Dumping                        2
Illegal Parking                        2
Leak                                   2
Litter Can                             2
Litter Can, Public                     2
Potholes                               2
Real Estate Tax                        2
Refuse Violations                      2
Request New Sign                       2
Thank You                              2
Water/Drinking Fountains               2
Name: request_type_id, dtype: int64

396224（1个）,464(7个）
33748 （8个）, 16214 （10个）
370（215个）, 14565 (7个）
468456（17个），362488（2个）
828（187个），160776（5个）
417（207个），160754（2个）
394（22个）,525（16个）
429（7个）,478（3个）
833（41个）,478（3个）
484（783个），160741（19个）
14568（221个），373（21个）
512（201个），160755（4个）
270186（28个），491（28个）
12074（261个）,255453（49个）
40178（15个），827（14个）

In [147]:
correct_mapping = {
    'Crosswalk, New': 464,
    'Dumpster': 16214,
    'Earned Income Tax': 370,
    'Electronic/Hazardous Waste Disposal': 468456,
    'Illegal Dumping': 828,
    'Illegal Parking': 417,
    'Leak': 394,
    'Litter Can': 429,
    'Litter Can, Public': 833,
    'Potholes': 484,
    'Real Estate Tax': 14568,
    'Refuse Violations': 512,
    'Request New Sign': 270186,
    'Thank You': 12074,
    'Water/Drinking Fountains': 40178
}

In [148]:
def correct_request_type_id(row):
    correct_id = correct_mapping.get(row['request_type_name'])
    if correct_id is not None and row['request_type_id'] != correct_id:
        return correct_id
    else:
        return row['request_type_id']
        
data['request_type_id_corrected'] = data.apply(correct_request_type_id, axis=1)

In [149]:
name_id_mapping = data.groupby('request_type_name')['request_type_id_corrected'].nunique()
name_id_mapping[name_id_mapping != 1]

Series([], Name: request_type_id_corrected, dtype: int64)

## (8) request_type

In [150]:
data['request_type_name'].nunique()

345

In [151]:
data['request_type_name'].unique()

array(['Replace/Repair a Sign', 'Permits, Licenses and Inspections',
       'Litter, Public Property', 'Potholes', 'Illegal Dumping',
       'Leaves/Street Cleaning', 'Retaining Wall',
       'Tree Fallen Across Road', 'Public Right of Way',
       'Crosswalk and Street Markings, Maintenance', 'Request New Sign',
       'Refuse Violations', 'DO NOT USE (Vacant and Open Building)',
       'Missed Refuse Pick Up', 'Pruning (city tree)', 'Field',
       'Dead tree (Public property)', 'Street Cleaning/Sweeping',
       'Root prune', 'Overgrowth', 'Abandoned Vehicle (parked on street)',
       'Tree Fallen Across Sidewalk', 'ADA Ramp, Installation',
       'Curb/Request for Asphalt Windrow', 'Tree Removal',
       'Paving Request', 'Dumpster (on Street)', 'Planting',
       'Utility Cut - PWSA', 'Pedestrian Signal Request', 'Road',
       'Sidewalk/Curb/ADA Ramp Maintenance', 'Curb Cuts',
       'Stump Grind/Removal', 'Utility Pole',
       'Traffic or Pedestrian Signal, Request', 'Traffic 

In [153]:
type_counts = data['request_type_name'].value_counts(normalize=True)

# 查看前60个高频类别
type_counts.head(60)

request_type_name
Weeds/Debris                            0.096412
Potholes                                0.083590
Missed Refuse Pick Up                   0.051264
Snow/Ice removal                        0.039506
Building Maintenance                    0.035799
Refuse Violations                       0.032379
Abandoned Vehicle (parked on street)    0.029345
Illegal Parking                         0.026050
Litter, Public Property                 0.022017
Street Light - Repair                   0.021898
Missed Recycling Pick Up                0.021438
Replace/Repair a Sign                   0.017128
Building Without a Permit               0.014615
Overgrowth                              0.013458
Paving Request                          0.011865
Pruning (city tree)                     0.010988
City Source (CDBG)                      0.010807
Early Set Out                           0.010505
Excessive Noise/Disturbances            0.009936
Dead Animal                             0.009832
St

In [154]:
# 定义阈值（例如占比<0.1%的类别视为低频）
threshold = 0.001
low_freq_types = type_counts[type_counts < threshold].index.tolist()
low_freq_types

['Street Light - Request',
 'Couch on Porch',
 'Nuisance Bar',
 'Bus Shelter',
 'Premature Requests',
 'Trail Maintenance',
 'Port A Potty',
 'Curb Painting, New',
 'No Parking Variance',
 'Concern',
 'Abandoned Vehicle',
 'Guide Rail',
 'Handicapped Parking Application',
 'Stump Grind/Removal',
 'Real Estate Tax',
 'Leaves, Grass or Other Yard Debris',
 'Replace/Repair Sign',
 'Pedestrian Signal Maintenance',
 'Panhandling',
 'Court, Basketball or Tennis',
 'Litter Can',
 'Electrical Violation',
 'Brick or Block Repair',
 'Tenant/Landlord Problems',
 'Signs, Advertising or Political',
 'Graffiti in Right of Way',
 'Commission on Human Relations',
 'Special Pick Up',
 'Fire Prevention',
 'Overcrowding',
 'Ethics Office',
 'Blocked or Closed Trails',
 'Retaining Wall (Private Property)',
 'CitiParks Programs',
 'Crossing Guards',
 'Thank You - Police',
 'Planting',
 'Recycling Violation',
 'PWSA Billing or Shut Off',
 'Unpermitted Land Operations',
 'Junk Vehicles (Private Property)',
 

In [ ]:
# 合并低频类别为"Other"
df['request_type_grouped'] = df['request_type_name'].apply(
    lambda x: x if x not in low_freq_types else 'Other'
)

In [ ]:
# 步骤1：合并低频类别
type_counts = df['request_type_name'].value_counts(normalize=True)
low_freq_types = type_counts[type_counts < 0.001].index.tolist()
df['request_type_grouped'] = df['request_type_name'].apply(
    lambda x: x if x not in low_freq_types else 'Other'
)

# 步骤2：合并相似类别
category_merge_map = {
    'Potholes': 'Road_Issues',
    'Pothole Repair': 'Road_Issues',
    'Illegal Dumping': 'Waste_Issues',
    'Garbage Pickup': 'Waste_Issues'
}
df['request_type_grouped'] = df['request_type_grouped'].replace(category_merge_map)

# 步骤3：One-Hot编码
origin_dummies = pd.get_dummies(df['origin'], prefix='origin', dtype=int)
df = pd.concat([df, origin_dummies], axis=1)


## (9) time

In [ ]:
data = data[~data['request_type_name'].str.contains(r'\(DO NOT USE\)', case=False, na=False)]

In [193]:
data['request_type_name'].nunique()

333

In [202]:
data['request_type_name'].unique()

array(['Replace/Repair a Sign', 'Permits, Licenses and Inspections',
       'Litter, Public Property', 'Potholes', 'Illegal Dumping',
       'Leaves/Street Cleaning', 'Retaining Wall',
       'Tree Fallen Across Road', 'Public Right of Way',
       'Crosswalk and Street Markings, Maintenance', 'Request New Sign',
       'Refuse Violations', 'DO NOT USE (Vacant and Open Building)',
       'Missed Refuse Pick Up', 'Pruning (city tree)', 'Field',
       'Dead tree (Public property)', 'Street Cleaning/Sweeping',
       'Root prune', 'Overgrowth', 'Abandoned Vehicle (parked on street)',
       'Tree Fallen Across Sidewalk', 'ADA Ramp, Installation',
       'Curb/Request for Asphalt Windrow', 'Tree Removal',
       'Paving Request', 'Dumpster (on Street)', 'Planting',
       'Utility Cut - PWSA', 'Pedestrian Signal Request', 'Road',
       'Sidewalk/Curb/ADA Ramp Maintenance', 'Curb Cuts',
       'Stump Grind/Removal', 'Utility Pole',
       'Traffic or Pedestrian Signal, Request', 'Traffic 

In [194]:
closed_requests = data[data['closed_date_et'].notna()].copy()
open_requests = data[data['closed_date_et'].isna()].copy()

In [195]:
data.shape

(799566, 32)

In [196]:
print(closed_requests.shape,closed_requests.shape[0]/800000)
print(open_requests.shape,open_requests.shape[0]/800000)

(710557, 32) 0.88819625
(89009, 32) 0.11126125


### i. Non-closed request

In [197]:
status_dist = open_requests['status_name'].value_counts(normalize=True).mul(100).round(2)
print(f"status distribution（%）:\n{status_dist.to_string()}")

status distribution（%）:
status_name
in progress    51.40
open           48.08
on hold         0.52


In [198]:
for status in ['open', 'in progress', 'on hold']:
    status_subset = open_requests[open_requests['status_name'] == status]
    top_types = status_subset['request_type_name'].value_counts().head(10)
    print(f"\n{status}: top 5 request types")
    print(top_types.to_string())


open: top 5 request types
request_type_name
Abandoned Vehicle (parked on street)    7866
Illegal Parking                         2477
SPIN (Stand Up) Scooters                2211
Drug Enforcement                        1924
Dashcam                                 1768
City Source (CDBG)                      1390
Speeding                                1151
City Owned Property Maintenance          945
Snow/Ice removal                         919
Health Hazard                            738

in progress: top 5 request types
request_type_name
Weeds/Debris                            11122
Building Maintenance                     6430
Abandoned Vehicle (parked on street)     1928
Building Without a Permit                1871
Illegal Parking                          1609
Vacant Building                          1584
Broken Sidewalk                          1053
Pruning (city tree)                       748
Patrol                                    690
Excessive Noise/Disturbances           

### ii. Closed request

In [199]:
closed_top = closed_requests['request_type_name'].value_counts().head(10)
print(f"closed requests top 10:\n{closed_top.to_string()}")

closed requests top 10:
request_type_name
Potholes                    66601
Weeds/Debris                65685
Missed Refuse Pick Up       40335
Snow/Ice removal            30354
Refuse Violations           25238
Building Maintenance        22108
Street Light - Repair       17263
Missed Recycling Pick Up    16951
Litter, Public Property     16869
Illegal Parking             16718


In [200]:
closed_requests['time_taken'] = (closed_requests['closed_date_et'] - closed_requests['create_date_et']).dt.total_seconds() / 3600
closed_requests['time_taken']

151382      222.800000
173037     5069.000000
196183     8622.066667
111096      554.850000
413448    21049.116667
              ...     
799995       49.983333
799997        0.250000
799994       17.983333
799993       53.166667
799999     2015.050000
Name: time_taken, Length: 710557, dtype: float64

In [201]:
time_stats = closed_requests.groupby('request_type_name')['time_taken'].agg(
    ['count', 'mean', 'median', 'min', 'max', 'std']
).sort_values('mean', ascending=False)

# 显示处理时间最长的类型
print("\n平均处理时间最长的10种请求:")
print(time_stats.head(10).to_string())

# 显示处理时间最短的类型
print("\n平均处理时间最短的10种请求:")
print(time_stats[time_stats['count'] > 50].tail(10).to_string())  # 过滤低频类型

# 异常值检查（超过30天的请求）
long_requests = closed_requests[closed_requests['time_taken'] > 720]  # 30天=720小时
print(f"\n超过30天处理的请求数量: {len(long_requests)}")
print("这些请求的类型分布:")
print(long_requests['request_type_name'].value_counts().head(10).to_string())


平均处理时间最长的10种请求:
                                                 count          mean        median           min           max           std
request_type_name                                                                                                           
Retaining Wall (Public)                            197  24692.589932   7469.116667      0.500000  77955.416667  26245.410870
Bicycle/Pedestrian/Trail - Network Improvements      2  22467.908333  22467.908333  22442.800000  22493.016667     35.508546
Water Runoff in ROW                                150  18896.862556  18671.408333      0.000000  43938.000000  12142.269429
Sidewalk/Curb/ADA Ramp Maintenance                2365  17848.397364  19323.350000      0.000000  84715.950000  13362.061815
Traffic Signals Surtrac                             90  16050.143704  18050.341667      0.050000  29014.466667   8419.948796
County Maintenance                                 275  15544.423273  10782.583333      0.183333  53788.6666

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt

time_stats = closed_requests.groupby('request_type_name')['time_taken'].agg(
    ['count', 'median', 'mean']
)


# 选取Top10长平均时间类型
top10_slow = time_stats.nlargest(10, 'mean').index.tolist()
subset = closed_requests[closed_requests['request_type_name'].isin(top10_slow)]

# 绘制箱线图
plt.figure(figsize=(12,6))
sns.boxplot(data=subset, x='request_type_name', y='time_taken')
plt.xticks(rotation=90)
plt.ylim(0, 5000)  # 聚焦主要分布区间
plt.title('Top10长平均时间类型的处理时间分布')
plt.show()


NameError: name 'closed_requests' is not defined

In [40]:
start_date = pd.to_datetime('2020-01-01')
end_date = pd.to_datetime('2024-12-31')

filtered_data = data[(data['create_date_et'] >= start_date) & (data['create_date_et'] <= end_date)].sort_values(by=['create_date_et'])

In [42]:
filtered_data['create_date_et']

150119   2020-01-01 09:53:00
150437   2020-01-01 09:55:00
11190    2020-01-01 10:44:00
554165   2020-01-01 11:02:00
201883   2020-01-01 11:31:00
                 ...        
799997   2024-12-18 09:28:00
799994   2024-12-18 09:32:00
799993   2024-12-18 09:39:00
799990   2024-12-18 09:40:00
799999   2024-12-18 09:41:00
Name: create_date_et, Length: 438120, dtype: datetime64[ns]

In [43]:
filtered_data.shape

(438120, 21)

# Sparse-Column-Identification

In [44]:
data.nunique(axis=0)#.head(10)

_id                  800000
group_id             800000
num_requests             40
parent_closed             2
status_name               4
status_code               4
dept                     62
request_type_name       345
request_type_id         336
create_date_et       629058
last_action_et       462677
closed_date_et       411476
origin                   10
city                     25
neighborhood             92
council_district          9
ward                     33
police_zone               6
latitude             480172
longitude            495595
geo_accuracy              5
dtype: int64

In [45]:
data.nunique()/len(data)*100

_id                  100.000000
group_id             100.000000
num_requests           0.005000
parent_closed          0.000250
status_name            0.000500
status_code            0.000500
dept                   0.007750
request_type_name      0.043125
request_type_id        0.042000
create_date_et        78.632250
last_action_et        57.834625
closed_date_et        51.434500
origin                 0.001250
city                   0.003125
neighborhood           0.011500
council_district       0.001125
ward                   0.004125
police_zone            0.000750
latitude              60.021500
longitude             61.949375
geo_accuracy           0.000625
dtype: float64

In [46]:
data.values

array([[151383, 836, 1, ..., 40.4746166, -79.96131760000003, 'EXACT'],
       [173038, 837, 1, ..., nan, nan, 'REDACTED'],
       [196184, 838, 1, ..., 40.424091, -79.996103, 'EXACT'],
       ...,
       [800128, 911529, 1, ..., 40.4523762, -80.0191839, 'EXACT'],
       [800125, 911530, 1, ..., 40.4566788, -80.0125067, 'APPROXIMATE'],
       [800134, 911531, 1, ..., 40.426867, -80.002561, 'EXACT']],
      dtype=object)

In [89]:
from sklearn.feature_selection import VarianceThreshold

data_value = data.values

X = data_value[:, :-1]
y = data_value[:, -1]

print(X.shape, y.shape)

vt = VarianceThreshold()

X_sel = vt.fit_transform(X)
print(X_sel.shape)

(800000, 20) (800000,)


ValueError: could not convert string to float: 't'